In [1]:
import numpy as np
import os
import time
import matplotlib.pyplot as plt
import torch as T
import pandas as pd
from scipy import signal
import math


def fft_signal(data_, iota):
    '''def safe_log10(x, eps=1e-10):
        return np.log10(np.where(x > eps, x, eps))'''
    
    N = 2 ** np.ceil(np.log2(len(data_) * 4))
    f, x = signal.periodogram(data_, window='hann', nfft=N, fs=1e9 * iota, scaling='spectrum')
    f = f / 1e6
    epsilon = 1e-10
    x = 10 * (np.log10((x + epsilon)/ 50e-3))
    return [f, x]

def get_data_in_array(dataframe, header_name, delim=True, size=400):
    _results_array = np.zeros([len(dataframe), size])
    for i in range(len(dataframe)):
        _results = dataframe[header_name].iloc[i].strip()[1:-1]
        _results = _results.split(",") if delim else _results.split()
        np_array = [float(value.strip()) for value in _results]
        _results_array[i] = np.array(np_array)[:400]
    return _results_array


def get_datasets(dataset):
    x = get_data_in_array(dataframe=dataset, header_name='arcsin_results', delim=True).astype(float)
    y = get_data_in_array(dataframe=dataset, header_name='test_data', delim=True).astype(float)

    x = np.resize(x, (x.shape[0], 400))
    y = np.resize(y, (y.shape[0], 400))

    data_tensor = T.tensor(x, dtype=T.float32)
    targets_tensor = T.tensor(y, dtype=T.float32)

    testing_dataset = T.utils.data.TensorDataset(data_tensor, targets_tensor)
    testing_loader = T.utils.data.DataLoader(testing_dataset, batch_size=1, shuffle=False)
    #freqs = dataset['Frequencies'].reset_index(drop=True)

    return testing_loader

def plot_fft_per_frequency(model_path='./best_models/best_model.pt', csv_path='power_selected.csv'):
    model = T.jit.load(model_path)
    model.eval()
    device = T.device('cuda' if T.cuda.is_available() else 'cpu')
    model.to(device)

    df = pd.read_csv(csv_path, skiprows=[0], names=[
        'Numbers', 'Frequencies', 'Powers', 'arcsin_results', 'test_data', 'expected_recovered_freq'])

    freq_list = sorted(df['Frequencies'].unique())

    for freq in freq_list:
        df_freq = df[df['Frequencies'] == freq].reset_index(drop=True)
        loader = get_datasets(df)

        all_pred_fft = []
        all_target_fft = []
        plt.figure(figsize=(12, 6))

        for i, (data, target) in enumerate(loader):
            data = data.to(device)
            target = target.to(device)
            with T.no_grad():
                output = model(data.unsqueeze(0)).squeeze().cpu().numpy()
                target = target.squeeze().cpu().numpy()

            f_target, x_target = fft_signal(target, iota=2)
            
            f_pred, x_pred = fft_signal(output, iota=2)

            all_target_fft.append(x_target)
            all_pred_fft.append(x_pred)

        # Convert to numpy arrays for averaging
        all_target_fft = np.array(all_target_fft)
        all_pred_fft = np.array(all_pred_fft)

        # Average over 1500 samples
        #avg_target = np.mean(all_target_fft, axis=0)
        #avg_pred = np.mean(all_pred_fft, axis=0)

        # Use frequency axis from first sample (they're all same)
        #f_axis, _ = fft_signal(target, iota=2)

        # Plot
        plt.plot(f_target, x_target, label='Average Target', color='red')
        plt.plot(f_pred, x_pred, label='Average Predicted', color='green', linestyle='--')
        plt.title("FFT Spectrum Error (dB)")
        plt.xlabel('Frequency (MHz)')
        plt.ylabel('Power Spectrum (dB)')
        plt.title(f'Average FFT Spectrum\nFrequency: {freq:.5f} GHz (All Powers)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()
plot_fft_per_frequency()

: 